# Урок 16 · Проект «Классификатор моих фото»

Сегодня ты соберёшь **настоящий продукт**: обучишь сеть узнавать твои категории на ТВОИХ фото
и получишь ссылку, которую можно отправить родителям с телефона.

**Важно:** включи GPU → Среда выполнения → Сменить среду выполнения → T4 GPU. Так обучение займёт секунды, а не минуты.

> План: 1) данные → 2) аугментация → 3) дообучаем MobileNet → 4) веб-приложение Gradio.

## Шаг 0. Проверка окружения

Запусти — убедись, что всё на месте. Если увидишь GPU — отлично.

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", "есть 🚀" if tf.config.list_physical_devices('GPU') else "нет (будет медленнее, но сработает)")

## Шаг 1. Загружаем данные — ДВА пути

Выбери СВОЙ путь:

- **Путь А** — у тебя есть свои фото → запусти клетку А, нажми кнопку и выбери файлы.
- **Путь Б** — фото нет или не успел → запусти клетку Б, возьмём запасной датасет (кошки/собаки).

Делай ТОЛЬКО ОДИН путь. Потом иди к шагу 2.

### 🅰️ Путь А. Мои фото через кнопку загрузки

Как подготовить фото ДО загрузки:
1. Выбери 2 категории (например `cat` и `dog`).
2. Собери по 20–30 фото каждой.
3. **Важно:** переименуй файлы так, чтобы имя начиналось с категории и подчёркивания:
   `cat_1.jpg`, `cat_2.jpg`, ..., `dog_1.jpg`, `dog_2.jpg`
   (по первому слову до `_` мы поймём категорию).

Запусти клетку, нажми «Выбрать файлы» и отметь все фото сразу.

In [ ]:
from google.colab import files
import os, shutil

# очищаем старую папку, если запускаешь второй раз
if os.path.exists("data"): shutil.rmtree("data")
os.makedirs("data", exist_ok=True)

print("Нажми кнопку ниже и выбери ВСЕ свои фото (имена вида cat_1.jpg, dog_1.jpg):")
uploaded = files.upload()   # <- откроется кнопка выбора файлов

# раскладываем файлы по папкам-категориям на основе имени до "_"
for fname in uploaded.keys():
    category = fname.split("_")[0]          # cat_1.jpg -> "cat"
    folder = os.path.join("data", category)
    os.makedirs(folder, exist_ok=True)
    shutil.move(fname, os.path.join(folder, fname))

print("Готово! Категории:", os.listdir("data"))
for c in os.listdir("data"):
    print(" ", c, "—", len(os.listdir(os.path.join("data", c))), "фото")

### 🅱️ Путь Б. Запасной датасет (если своих фото нет)

Скачиваем маленький готовый набор кошки/собаки. Ничего выбирать не нужно.

In [ ]:
import tensorflow as tf, os, shutil, pathlib

# официальный маленький датасет кошки/собаки от Google
url = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
path = tf.keras.utils.get_file("cats_and_dogs.zip", origin=url, extract=True)
src = os.path.join(os.path.dirname(path), "cats_and_dogs_filtered", "train")

# копируем в папку data/cat и data/dog (берём по 30 фото для скорости)
if os.path.exists("data"): shutil.rmtree("data")
for cls_src, cls_dst in [("cats","cat"), ("dogs","dog")]:
    dst = os.path.join("data", cls_dst); os.makedirs(dst, exist_ok=True)
    imgs = sorted(os.listdir(os.path.join(src, cls_src)))[:30]
    for im in imgs:
        shutil.copy(os.path.join(src, cls_src, im), os.path.join(dst, im))

print("Запасной датасет готов. Категории:", os.listdir("data"))
for c in os.listdir("data"):
    print(" ", c, "—", len(os.listdir(os.path.join("data", c))), "фото")

## Шаг 2. Смотрим на свои данные

Первое правило: всегда посмотри на данные глазами, прежде чем обучать.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

categories = sorted(os.listdir("data"))    # например ['cat', 'dog']
print("Мои категории:", categories)

# показываем по 3 фото из каждой категории
plt.figure(figsize=(9, 3*len(categories)))
i = 1
for c in categories:
    files_in_c = os.listdir(os.path.join("data", c))[:3]
    for f in files_in_c:
        img = Image.open(os.path.join("data", c, f))
        plt.subplot(len(categories), 3, i)
        plt.imshow(img); plt.title(c); plt.axis("off")
        i += 1
plt.tight_layout(); plt.show()

## Шаг 3. Готовим данные для сети

Сеть MobileNet ждёт картинки одного размера (160×160) и пачками (батчами).
`image_dataset_from_directory` сам прочитает папки и назначит метки по названиям папок.

In [ ]:
import tensorflow as tf

IMG_SIZE = (160, 160)

# создаём обучающий и проверочный наборы (80% / 20%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data", validation_split=0.2, subset="training", seed=42,
    image_size=IMG_SIZE, batch_size=8)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data", validation_split=0.2, subset="validation", seed=42,
    image_size=IMG_SIZE, batch_size=8)

class_names = train_ds.class_names       # запоминаем названия категорий по порядку
print("Порядок категорий:", class_names)

## Шаг 4. Аугментация — из мало фото делаем много

30 фото сети маловато — она может просто их запомнить (переобучение).
Аугментация слегка поворачивает и отражает каждую картинку, будто мы смотрим на объект с разных сторон.

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation
from tensorflow.keras.models import Sequential

augment = Sequential([
    RandomFlip("horizontal"),   # случайное зеркальное отражение
    RandomRotation(0.1),        # лёгкий случайный поворот
])

# посмотрим, что делает аугментация с ОДНОЙ картинкой
for images, _ in train_ds.take(1):
    plt.figure(figsize=(9,3))
    first = images[0]
    for i in range(3):
        aug = augment(tf.expand_dims(first, 0))[0]
        plt.subplot(1,3,i+1); plt.imshow(aug.numpy().astype("uint8")); plt.axis("off")
    plt.suptitle("Одно фото — три варианта"); plt.show()

## Шаг 5. Дообучаем MobileNetV2 (transfer learning)

Берём «готового эксперта» MobileNetV2, **замораживаем** его знания и добавляем сверху свою маленькую «голову»
под наши категории. Это и есть transfer learning.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input
from tensorflow.keras.models import Model

num_classes = len(class_names)    # сколько у нас категорий

# готовый эксперт, обученный на миллионах картинок ImageNet
base = MobileNetV2(input_shape=(160,160,3), include_top=False, weights="imagenet")
base.trainable = False            # ЗАМОРАЖИВАЕМ — не стираем знания эксперта

# собираем модель: аугментация -> подготовка -> эксперт -> наша голова
inputs = Input(shape=(160,160,3))
x = augment(inputs)
x = preprocess_input(x)           # MobileNet ждёт особую подготовку входа
x = base(x, training=False)
x = GlobalAveragePooling2D()(x)
outputs = Dense(num_classes, activation="softmax")(x)   # число выходов = число категорий
model = Model(inputs, outputs)

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
print("Модель собрана. Категорий:", num_classes)

In [ ]:
# обучаем: 10 проходов по данным. С GPU это очень быстро.
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

**❓ Вопрос.** Посмотри на `accuracy` и `val_accuracy` в последней эпохе.
Если `accuracy` высокая, а `val_accuracy` низкая — это признак переобучения. Что это значит для твоих фото?

## Шаг 6. Проверяем модель глазами

Прогоним несколько проверочных картинок и посмотрим, что предсказала сеть.

In [ ]:
import numpy as np

for images, labels in val_ds.take(1):
    preds = model.predict(images)
    plt.figure(figsize=(10,5))
    for i in range(min(6, len(images))):
        plt.subplot(2,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        guess = class_names[np.argmax(preds[i])]
        truth = class_names[labels[i]]
        color = "green" if guess == truth else "red"
        plt.title("сеть: "+guess, color=color); plt.axis("off")
    plt.tight_layout(); plt.show()

## Шаг 7. Веб-приложение через Gradio 🎉

Финал. Оборачиваем модель в веб-страницу с публичной ссылкой. `share=True` — и ссылку можно открыть с любого телефона.

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image

def classify(img):
    img = Image.fromarray(img).resize((160,160))       # приводим к нужному размеру
    arr = np.expand_dims(np.array(img).astype("float32"), axis=0)
    pred = model.predict(arr)[0]
    # возвращаем словарь {категория: вероятность}
    return {class_names[i]: float(pred[i]) for i in range(len(class_names))}

demo = gr.Interface(
    fn=classify,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=len(class_names)),
    title="Мой классификатор фото",
    description="Загрузи картинку — модель угадает категорию")

demo.launch(share=True)     # <- в выводе появится ссылка вида https://xxxxx.gradio.live

**Отправь ссылку родителям!** Это твой первый живой продукт в интернете.

> Ссылка `gradio.live` живёт пока открыт ноутбук (72 часа). Для постоянной ссылки есть Hugging Face Spaces — об этом на уроке 20.

---
## 🎯 Задания

### 🟢 Базовый
Собери классификатор из 2 категорий, обучи, запусти Gradio, проверь на НОВОМ фото (которого не было в обучении). Отправь ссылку другу.

### 🟡 Продвинутый
Сделай классификатор из 3 категорий. Обучи один раз С аугментацией и один раз БЕЗ (убери слой `augment` из модели). Сравни `val_accuracy`. Помогла ли аугментация на маленьком датасете?

### ⭐ Со звёздочкой
Найди фото, на котором модель уверенно ошибается. Почему так вышло? Что общего у ошибочных случаев (освещение, ракурс, фон)? Запиши гипотезу и проверь: добавь 5 таких фото в обучение и переобучи.

## Мини-итог

- Transfer learning — это ...
- Заморозка базы (`base.trainable = False`) нужна, потому что ...
- `share=True` в Gradio делает ...

> Ты прошёл путь от «нейрон — это сумма» до работающего веб-приложения на СВОИХ данных. Это уже настоящий проект.